<center>
<img src="https://www.udd.cl/dircom/web/udd/facultades/Ingenieria.png" width="420px">

# **Segundo Avance de Proyecto**
## **¿Existe una relación entre la desigualdad socioeconómica y el rendimiento escolar?**

| | |
|---|---|
| **Integrantes** | Nicolás Bravo y Luis Felipe Cáceres |
| **Curso** | Análisis de Datos e Inferencia Estadística |
| **Sección** | 1 |
| **Profesor** | Cristian García |
| **Ayudante** | Benjamín Bennet |
| **Fecha de entrega** | 08 de mayo de 2026 |

</center>

---
# 1. Introducción y pregunta de investigación

La desigualdad socioeconómica es uno de los factores más estudiados en relación con el rendimiento escolar. En Chile, el sistema educativo clasifica a los establecimientos según el nivel socioeconómico predominante de sus estudiantes (**Grupo Socioeconómico, GSE**), lo que permite analizar si existen brechas sistemáticas en los resultados académicos según este indicador.

**Pregunta de investigación:**  
> **¿Existe una relación entre la desigualdad socioeconómica y el rendimiento escolar en establecimientos educacionales chilenos?**

**Hipótesis general:**  
Los establecimientos con un grupo socioeconómico más alto obtendrán puntajes promedio SIMCE significativamente mayores que los establecimientos de GSE más bajo.

Para responder esta pregunta se utilizan dos bases de datos oficiales del año 2024: los resultados SIMCE de 2° medio (como medida del rendimiento escolar) y los Indicadores de Desarrollo Personal y Social (IDPS, como variables complementarias del contexto escolar).

---
# 2. Carga de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy import stats
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid")

# Paletas corporativas del proyecto
PALETA_GSE       = ["#AED6F1", "#5DADE2", "#2E86C1", "#1A5276", "#0D2B45"]
PALETA_GSE_VERDE = ["#A9DFBF", "#52BE80", "#1E8449", "#145A32", "#0B3D21"]
COLOR_LECT       = "#2471A3"
COLOR_MATE       = "#117A65"
ORDEN_GSE        = ["Bajo", "Medio bajo", "Medio", "Medio alto", "Alto"]

print("Librerías cargadas correctamente.")

---
# 3. Descripción del dataset y carga de datos

Se utilizan dos bases de datos del año 2024, ambas a nivel de establecimiento (RBD):

| Base | Fuente | Descripción |
|------|--------|-------------|
| **SIMCE 2° medio 2024** | Agencia de Calidad de la Educación | Puntajes promedio de lectura y matemática por establecimiento |
| **IDPS 2° medio 2024** | Agencia de Calidad de la Educación | Indicadores de desarrollo personal y social por establecimiento |

**Unidad de análisis:** establecimiento educacional (RBD).  
**Variable dependiente:** `promedio_simce` (promedio de lectura y matemática).  
**Variable independiente principal:** `grupo_socioeconomico` (GSE).  
**Variables de control:** indicadores IDPS, dependencia administrativa, zona urbana/rural.

In [ ]:
def buscar_archivo(nombre):
    rutas = [Path("."), Path("basesDeDatos/pregunta_2"), Path("../basesDeDatos/pregunta_2")]
    for ruta in rutas:
        p = ruta / nombre
        if p.exists():
            return p
    raise FileNotFoundError(f"No se encontró: {nombre}")

simce = pd.read_csv(buscar_archivo("simce2m2024_rbd_preliminar.csv"), sep=";", encoding="latin1")
idps  = pd.read_csv(buscar_archivo("idps2M2024_rbd_preliminar.csv"),  sep=";", encoding="latin1")

print(f"SIMCE:  {simce.shape[0]:,} filas × {simce.shape[1]} columnas")
print(f"IDPS:   {idps.shape[0]:,} filas × {idps.shape[1]} columnas")
display(simce.head(3))

---
# 4. Glosario de variables relevantes

| Variable | Descripción |
|----------|-------------|
| `rbd` | Identificador único del establecimiento educacional |
| `grupo_socioeconomico` (GSE) | Clasificación socioeconómica del establecimiento: Bajo, Medio bajo, Medio, Medio alto, Alto |
| `prom_lect2m_rbd` | Puntaje promedio SIMCE de Lectura en 2° medio 2024 |
| `prom_mate2m_rbd` | Puntaje promedio SIMCE de Matemática en 2° medio 2024 |
| `promedio_simce` | Promedio entre lectura y matemática (variable creada) |
| `idps_autoestima` | Indicador IDPS: autoestima académica y motivación escolar |
| `idps_convivencia` | Indicador IDPS: clima de convivencia escolar |
| `idps_habitos` | Indicador IDPS: hábitos de vida saludable |
| `idps_participacion` | Indicador IDPS: participación y formación ciudadana |
| `dependencia` | Tipo administrativo: Municipal, Part. subvencionado, Part. pagado, SLEP |
| `zona` | Clasificación territorial: Urbano / Rural |

---
# 5. Limpieza y preparación de datos

## 5.1 Selección de variables y codificación

In [ ]:
# Selección de columnas relevantes de SIMCE
columnas_simce = [
    "rbd", "nom_rbd", "nom_reg_rbd", "nom_com_rbd",
    "cod_depe2", "cod_grupo", "cod_rural_rbd",
    "nalu_lect2m_rbd", "nalu_mate2m_rbd",
    "prom_lect2m_rbd", "prom_mate2m_rbd"
]
simce_sel = simce[columnas_simce].copy()

# Diccionarios de traducción
mapa_gse = {1: "Bajo", 2: "Medio bajo", 3: "Medio", 4: "Medio alto", 5: "Alto"}
mapa_dep = {1: "Municipal", 2: "Part. subvencionado", 3: "Part. pagado", 4: "SLEP"}
mapa_zona = {1: "Urbano", 2: "Rural"}

simce_sel["grupo_socioeconomico"] = simce_sel["cod_grupo"].map(mapa_gse)
simce_sel["dependencia"]          = simce_sel["cod_depe2"].map(mapa_dep)
simce_sel["zona"]                 = simce_sel["cod_rural_rbd"].map(mapa_zona)

# Transformar IDPS: de formato largo a ancho (una fila por RBD)
idps_wide = (
    idps.pivot_table(index="rbd", columns="ind", values="prom", aggfunc="first")
    .reset_index()
)
idps_wide.columns.name = None
idps_wide = idps_wide.rename(columns={
    "AM": "idps_autoestima",
    "CC": "idps_convivencia",
    "HV": "idps_habitos",
    "PF": "idps_participacion"
})

# Unión de bases por RBD
df = simce_sel.merge(idps_wide, on="rbd", how="left")

# Variable dependiente: promedio SIMCE
df["promedio_simce"] = df[["prom_lect2m_rbd", "prom_mate2m_rbd"]].mean(axis=1)

print(f"Dataset unido: {df.shape[0]:,} filas × {df.shape[1]} columnas")
display(df.head(3))

## 5.2 Valores ausentes

In [ ]:
vars_revisar = [
    "grupo_socioeconomico", "prom_lect2m_rbd", "prom_mate2m_rbd", "promedio_simce",
    "idps_autoestima", "idps_convivencia", "idps_habitos", "idps_participacion"
]

nulos = df[vars_revisar].isna().sum().to_frame("nulos")
nulos["porcentaje (%)"] = (nulos["nulos"] / len(df) * 100).round(2)
display(nulos)

**Decisión:** Se eliminan los registros sin `grupo_socioeconomico` o sin puntajes SIMCE, ya que son indispensables para responder la pregunta de investigación. Los valores nulos en los indicadores IDPS se mantienen en el dataset principal y solo se excluyen al construir el modelo de regresión.

In [ ]:
df_limpio = df.dropna(subset=["grupo_socioeconomico", "prom_lect2m_rbd", "prom_mate2m_rbd", "promedio_simce"]).copy()

print(f"Filas antes de limpieza : {df.shape[0]:,}")
print(f"Filas después de limpieza: {df_limpio.shape[0]:,}")
print(f"Filas eliminadas         : {df.shape[0] - df_limpio.shape[0]:,}")

## 5.3 Duplicados

In [ ]:
dup = df_limpio["rbd"].duplicated().sum()
print(f"RBD duplicados en el dataset limpio: {dup}")
print("No existen establecimientos repetidos." if dup == 0 else f"Atención: hay {dup} RBD duplicados.")

## 5.4 Detección de valores atípicos (outliers)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Detección de outliers — Puntajes SIMCE 2° Medio 2024",
             fontsize=14, fontweight="bold", y=1.02)

for ax, col, color, titulo in [
    (axes[0], "prom_lect2m_rbd",  COLOR_LECT,  "Puntaje Lectura"),
    (axes[1], "prom_mate2m_rbd",  COLOR_MATE,  "Puntaje Matemáticas"),
]:
    datos = df_limpio[col].dropna()
    q1, q3 = datos.quantile(0.25), datos.quantile(0.75)
    iqr = q3 - q1
    n_out = ((datos < q1 - 1.5*iqr) | (datos > q3 + 1.5*iqr)).sum()

    bp = ax.boxplot(datos, patch_artist=True, vert=True, widths=0.5,
                    flierprops=dict(marker="o", markerfacecolor="#E74C3C", markersize=4, alpha=0.6),
                    medianprops=dict(color="#E74C3C", linewidth=2.5))
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.65)

    ax.set_title(f"{titulo}\n({n_out} outliers detectados por IQR)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Puntaje promedio")
    ax.set_xticks([])

plt.tight_layout()
plt.show()

print("Nota: los outliers se mantienen en el análisis, ya que corresponden a establecimientos")
print("reales con puntajes extremos, no a errores de carga.")

---
# 6. Análisis Exploratorio de Datos (EDA)
## 6.1 Estadística descriptiva general

In [ ]:
vars_num = ["prom_lect2m_rbd", "prom_mate2m_rbd", "promedio_simce",
            "idps_autoestima", "idps_convivencia", "idps_habitos", "idps_participacion"]

tabla_desc = df_limpio[vars_num].describe().T
tabla_desc["rango IQR"] = df_limpio[vars_num].quantile(0.75) - df_limpio[vars_num].quantile(0.25)
display(tabla_desc[["count","mean","50%","std","min","25%","75%","max","rango IQR"]].round(2).rename(
    columns={"50%":"mediana","mean":"media","std":"desv. std","count":"N"}
))

**Interpretación:** El promedio SIMCE global oscila en torno a 250 puntos, con una desviación estándar que sugiere dispersión moderada. Los indicadores IDPS presentan rangos distintos, lo que refleja variabilidad en las dimensiones del desarrollo personal y social entre establecimientos.

## 6.2 Distribución de establecimientos por GSE

In [ ]:
tabla_gse = (
    df_limpio["grupo_socioeconomico"]
    .value_counts().reindex(ORDEN_GSE)
    .reset_index()
    .rename(columns={"grupo_socioeconomico": "GSE", "count": "Frecuencia"})
)
tabla_gse["Porcentaje (%)"] = (tabla_gse["Frecuencia"] / tabla_gse["Frecuencia"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(tabla_gse["GSE"], tabla_gse["Frecuencia"],
              color=PALETA_GSE, edgecolor="white", linewidth=0.8)

for bar, pct in zip(bars, tabla_gse["Porcentaje (%)"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f"{int(bar.get_height()):,}\n({pct}%)",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold")

ax.set_title("Distribución de Establecimientos por Grupo Socioeconómico",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Grupo Socioeconómico (GSE)", fontsize=11)
ax.set_ylabel("N° de establecimientos", fontsize=11)
ax.set_ylim(0, tabla_gse["Frecuencia"].max() * 1.22)
sns.despine()
plt.tight_layout()
plt.show()

display(tabla_gse)

**Interpretación:** La mayor concentración de establecimientos se encuentra en los grupos Medio bajo y Bajo, mientras que el grupo Medio alto es el menos representado. Esta distribución refleja la composición real del sistema educativo chileno, donde los establecimientos de GSE alto son una minoría.

## 6.3 Estadística descriptiva por GSE

In [ ]:
resumen_gse = (
    df_limpio.groupby("grupo_socioeconomico")[["prom_lect2m_rbd", "prom_mate2m_rbd", "promedio_simce"]]
    .agg(["count", "mean", "median", "std"])
    .reindex(ORDEN_GSE)
)

resumen_gse.columns = [f"{m} {v.split('_')[1] if 'lect' in v or 'mate' in v else 'simce'}" 
                        for v, m in resumen_gse.columns]
display(resumen_gse.round(2))

## 6.4 Histogramas de puntajes SIMCE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Distribución de Puntajes SIMCE 2° Medio 2024",
             fontsize=14, fontweight="bold", y=1.02)

for ax, col, color, titulo, etiq in [
    (axes[0], "prom_lect2m_rbd", COLOR_LECT, "Lectura",       "Puntaje lectura"),
    (axes[1], "prom_mate2m_rbd", COLOR_MATE, "Matemáticas",   "Puntaje matemáticas"),
]:
    datos = df_limpio[col].dropna()
    media, mediana = datos.mean(), datos.median()
    ax.hist(datos, bins=35, color=color, alpha=0.82, edgecolor="white", linewidth=0.5)
    ax.axvline(media,   color="tomato",   linewidth=2.2, linestyle="-",
               label=f"Media: {media:.1f}")
    ax.axvline(mediana, color="#E67E22",  linewidth=2,   linestyle="--",
               label=f"Mediana: {mediana:.1f}")
    ax.set_title(f"Puntaje SIMCE {titulo}", fontsize=12, fontweight="bold")
    ax.set_xlabel(etiq, fontsize=10)
    ax.set_ylabel("Frecuencia", fontsize=10)
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

**Interpretación:** Ambas distribuciones presentan forma aproximadamente normal con leve asimetría. La distribución de matemáticas muestra mayor dispersión que lectura, lo que anticipa diferencias más pronunciadas entre grupos socioeconómicos en esa prueba.

## 6.5 Boxplots por GSE — Lectura y Matemáticas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Puntajes SIMCE 2024 por Grupo Socioeconómico",
             fontsize=15, fontweight="bold", y=1.02)

for ax, col, paleta, titulo in [
    (axes[0], "prom_lect2m_rbd", PALETA_GSE,       "Lectura"),
    (axes[1], "prom_mate2m_rbd", PALETA_GSE_VERDE, "Matemáticas"),
]:
    color_median = "#E74C3C" if paleta == PALETA_GSE else "#E74C3C"
    edge_color   = "#1A5276"  if paleta == PALETA_GSE else "#145A32"

    sns.boxplot(
        data=df_limpio, x="grupo_socioeconomico", y=col,
        order=ORDEN_GSE, palette=paleta,
        width=0.55, linewidth=1.2,
        flierprops=dict(marker="o", markerfacecolor="#E74C3C",
                        markersize=3, alpha=0.45, linestyle="none"),
        medianprops=dict(color="#E74C3C", linewidth=2.5),
        ax=ax
    )
    medias = df_limpio.groupby("grupo_socioeconomico")[col].mean().reindex(ORDEN_GSE)
    ax.scatter(range(len(ORDEN_GSE)), medias.values, color="white", s=70,
               zorder=5, edgecolors=edge_color, linewidths=1.5, label="Media")

    ax.set_title(f"Puntaje SIMCE {titulo}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Grupo Socioeconómico", fontsize=10)
    ax.set_ylabel("Puntaje promedio", fontsize=10)
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

plt.tight_layout()
plt.show()

**Interpretación:** Se observa una relación positiva clara y consistente entre el GSE del establecimiento y el puntaje SIMCE, tanto en lectura como en matemáticas. La diferencia entre el grupo Bajo y el grupo Alto alcanza aproximadamente **56 puntos en lectura** y **90 puntos en matemáticas**, lo que evidencia una brecha académica sustancial asociada a la desigualdad socioeconómica. La mayor dispersión en el grupo Alto sugiere que factores adicionales influyen en el rendimiento incluso dentro de ese grupo.

## 6.6 Promedio SIMCE por GSE y prueba

In [ ]:
tabla_medias = (
    df_limpio.groupby("grupo_socioeconomico")[["prom_lect2m_rbd", "prom_mate2m_rbd"]]
    .mean().reindex(ORDEN_GSE)
    .rename(columns={"prom_lect2m_rbd": "Lectura", "prom_mate2m_rbd": "Matemáticas"})
)

x = np.arange(len(ORDEN_GSE))
ancho = 0.36

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - ancho/2, tabla_medias["Lectura"],      ancho,
            label="Lectura",      color=COLOR_LECT, alpha=0.88, edgecolor="white")
b2 = ax.bar(x + ancho/2, tabla_medias["Matemáticas"],  ancho,
            label="Matemáticas",  color=COLOR_MATE, alpha=0.88, edgecolor="white")

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1.5,
            f"{h:.1f}", ha="center", va="bottom", fontsize=8.5, color="#1a1a1a")

ax.set_title("Promedio SIMCE de Lectura y Matemáticas por Grupo Socioeconómico",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Grupo Socioeconómico", fontsize=11)
ax.set_ylabel("Puntaje promedio SIMCE", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(ORDEN_GSE, fontsize=10)
ax.legend(title="Prueba", fontsize=9)
ax.set_ylim(180, 320)
sns.despine()
plt.tight_layout()
plt.show()

display(tabla_medias.round(1))

**Interpretación:** El aumento progresivo de los puntajes promedio a medida que sube el GSE es consistente en ambas pruebas. La brecha es más pronunciada en matemáticas, donde la diferencia entre Bajo y Alto supera los 90 puntos. Este patrón sugiere que las condiciones socioeconómicas del entorno escolar se asocian con el rendimiento académico de manera sistemática.

## 6.7 Scatter: GSE vs promedio SIMCE

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for i, (gse, color) in enumerate(zip(ORDEN_GSE, PALETA_GSE)):
    sub = df_limpio[df_limpio["grupo_socioeconomico"] == gse]
    ax.scatter(sub["cod_grupo"] + np.random.uniform(-0.15, 0.15, len(sub)),
               sub["promedio_simce"],
               color=color, alpha=0.35, s=25, label=gse, edgecolors="none")

x_vals = df_limpio["cod_grupo"].dropna()
y_vals = df_limpio.loc[x_vals.index, "promedio_simce"]
m, b = np.polyfit(x_vals, y_vals, 1)
x_line = np.linspace(1, 5, 100)
ax.plot(x_line, m*x_line + b, color="tomato", linewidth=2.5,
        linestyle="--", label=f"Tendencia (pendiente = {m:.1f} pts/nivel)", zorder=5)

ax.set_title("Relación entre Grupo Socioeconómico y Promedio SIMCE\n(jitter aplicado para visualizar densidad)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Código GSE (1 = Bajo → 5 = Alto)", fontsize=11)
ax.set_ylabel("Promedio SIMCE", fontsize=11)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(ORDEN_GSE, fontsize=9)
ax.legend(fontsize=8, title="GSE")
sns.despine()
plt.tight_layout()
plt.show()

## 6.8 Heatmap de correlaciones

In [ ]:
VARS_CORR = ["cod_grupo", "prom_lect2m_rbd", "prom_mate2m_rbd", "promedio_simce",
              "idps_autoestima", "idps_convivencia", "idps_habitos", "idps_participacion"]

LABELS_CORR = ["GSE (código)", "Lectura", "Matemáticas", "Prom. SIMCE",
                "Autoestima (IDPS)", "Convivencia (IDPS)", "Hábitos (IDPS)", "Part. familia (IDPS)"]

corr_mat = df_limpio[VARS_CORR].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_mat, annot=True, fmt=".2f", cmap="RdYlBu_r",
    vmin=-1, vmax=1, linewidths=0.5, linecolor="white",
    square=True, annot_kws={"size": 10, "weight": "bold"},
    xticklabels=LABELS_CORR, yticklabels=LABELS_CORR, ax=ax
)
ax.set_title("Matriz de Correlación de Spearman\nPuntajes SIMCE e Indicadores IDPS 2024",
             fontsize=13, fontweight="bold", pad=14)
ax.set_xticklabels(LABELS_CORR, rotation=35, ha="right", fontsize=9)
ax.set_yticklabels(LABELS_CORR, rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

**Interpretación:** La correlación de Spearman entre el GSE y el promedio SIMCE es **r = 0.72**, lo que indica una relación positiva fuerte y estadísticamente significativa. Entre los indicadores IDPS, la convivencia escolar presenta la correlación más alta con los puntajes SIMCE, mientras que la participación familiar muestra una correlación negativa leve, lo que puede reflejar un mayor involucramiento de familias en contextos de menor rendimiento.

---
# 7. Test de hipótesis

## 7.1 Formulación

| Elemento | Detalle |
|----------|---------|
| **Variable dependiente** | `promedio_simce` (variable cuantitativa continua) |
| **Variable de comparación** | `grupo_socioeconomico` (5 grupos: Bajo → Alto) |
| **Nivel de significancia** | α = 0.05 |
| **H₀** | No existen diferencias en el promedio SIMCE entre los 5 grupos socioeconómicos |
| **H₁** | Al menos un grupo socioeconómico presenta un promedio SIMCE diferente |

## 7.2 Justificación del test

Como existen **más de dos grupos** a comparar, el test adecuado es el **ANOVA de un factor**. Antes de aplicarlo, se verifica el supuesto de homogeneidad de varianzas mediante el **test de Levene**. Si este supuesto se viola (p < 0.05), se utiliza la variante **ANOVA de Welch**, que es robusta ante varianzas heterogéneas y es la recomendada en este caso dado el tamaño muestral desigual entre grupos.

In [ ]:
grupos_anova = [
    df_limpio[df_limpio["grupo_socioeconomico"] == g]["promedio_simce"].dropna().values
    for g in ORDEN_GSE
]

# ── Test de Levene (supuesto de igualdad de varianzas) ──
lev_stat, lev_p = stats.levene(*grupos_anova, center="median")
print("=" * 55)
print("TEST DE LEVENE — Homogeneidad de varianzas")
print("=" * 55)
print(f"  Estadístico W : {lev_stat:.4f}")
print(f"  Valor p       : {lev_p:.2e}")
if lev_p < 0.05:
    print("  ► Se rechaza la igualdad de varianzas (p < 0.05).")
    print("    → Se usará ANOVA de Welch como test principal.")
else:
    print("  ► No se rechaza la igualdad de varianzas.")

In [ ]:
# ── ANOVA clásico ──
f_stat, p_anova = stats.f_oneway(*grupos_anova)
print("=" * 55)
print("ANOVA CLÁSICO (F de Fisher)")
print("=" * 55)
print(f"  Estadístico F : {f_stat:.4f}")
print(f"  Valor p       : {p_anova:.2e}")

# ── ANOVA de Welch (robusto a varianzas distintas) ──
welch = anova_oneway(grupos_anova, use_var="unequal")
print()
print("=" * 55)
print("ANOVA DE WELCH (recomendado por varianzas desiguales)")
print("=" * 55)
print(f"  Estadístico F : {float(welch.statistic):.4f}")
print(f"  Valor p       : {float(welch.pvalue):.2e}")

if welch.pvalue < 0.05:
    print()
    print("  ► Decisión: SE RECHAZA H₀ (p < 0.05).")
    print("  ► Conclusión: existen diferencias estadísticamente")
    print("    significativas en el promedio SIMCE entre los")
    print("    grupos socioeconómicos.")

**Interpretación del resultado:**  
Dado que el valor p del ANOVA de Welch es prácticamente cero (p < 0.001), se rechaza la hipótesis nula. Esto indica que **al menos un grupo socioeconómico presenta un promedio SIMCE significativamente distinto** a los demás. El estadístico F extremadamente alto (F > 776) refleja que la varianza *entre* grupos es mucho mayor que la varianza *dentro* de cada grupo, confirmando que el GSE es un factor fuertemente diferenciador del rendimiento escolar.

## 7.3 Comparaciones posteriores: Test de Tukey

In [ ]:
tukey = pairwise_tukeyhsd(
    endog=df_limpio["promedio_simce"].dropna(),
    groups=df_limpio.loc[df_limpio["promedio_simce"].notna(), "grupo_socioeconomico"],
    alpha=0.05
)
print(tukey)

In [ ]:
# Visualización del test de Tukey
fig, ax = plt.subplots(figsize=(10, 6))
tukey.plot_simultaneous(ax=ax, xlabel="Promedio SIMCE", ylabel="Grupo socioeconómico")
ax.set_title("Intervalos de confianza simultáneos — Test de Tukey (α = 0.05)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

**Interpretación:** El test de Tukey confirma que **todos los pares de grupos socioeconómicos presentan diferencias estadísticamente significativas** (p < 0.05). Es decir, no solo existe una diferencia global, sino que cada nivel del GSE se distingue claramente de los demás en términos de rendimiento SIMCE. Esto refuerza la hipótesis de que la desigualdad socioeconómica está asociada de forma estructural y graduada con el rendimiento escolar.

## 7.4 Correlación de Spearman: GSE y promedio SIMCE

In [ ]:
sp_r, sp_p = stats.spearmanr(
    df_limpio["cod_grupo"],
    df_limpio["promedio_simce"],
    nan_policy="omit"
)

print("=" * 55)
print("CORRELACIÓN DE SPEARMAN")
print("=" * 55)
print(f"  Coeficiente ρ : {sp_r:.4f}")
print(f"  Valor p       : {sp_p:.2e}")
print()
print("  Interpretación:")
print(f"  Existe una correlación positiva fuerte (ρ = {sp_r:.2f}) entre")
print("  el GSE y el promedio SIMCE. Al ser p < 0.001, esta relación")
print("  es estadísticamente significativa.")

---
# 8. Modelo de regresión lineal múltiple

## 8.1 Objetivo del modelo

El objetivo es estimar cuánto se asocia cada factor (GSE, indicadores IDPS, dependencia y zona) con el promedio SIMCE de un establecimiento, manteniendo constantes los demás factores.

**Variable dependiente:** `promedio_simce`  
**Variables independientes:**
- `cod_grupo`: nivel socioeconómico (1 = Bajo → 5 = Alto)  
- Indicadores IDPS: `idps_autoestima`, `idps_convivencia`, `idps_habitos`, `idps_participacion`  
- `C(cod_depe2)`: tipo de dependencia (variable categórica)  
- `C(cod_rural_rbd)`: zona urbana/rural (variable categórica)

In [ ]:
vars_modelo = [
    "promedio_simce", "cod_grupo", "cod_depe2", "cod_rural_rbd",
    "idps_autoestima", "idps_convivencia", "idps_habitos", "idps_participacion"
]

df_modelo = df_limpio.dropna(subset=vars_modelo).copy()
print(f"Observaciones usadas en el modelo: {df_modelo.shape[0]:,}")

formula = (
    "promedio_simce ~ cod_grupo "
    "+ idps_autoestima + idps_convivencia + idps_habitos + idps_participacion "
    "+ C(cod_depe2) + C(cod_rural_rbd)"
)

modelo = smf.ols(formula=formula, data=df_modelo).fit()
print(modelo.summary())

## 8.2 Tabla resumen del modelo

In [ ]:
nombres_bonitos = {
    "Intercept"           : "Intercepto",
    "cod_grupo"           : "GSE (código 1-5)",
    "idps_autoestima"     : "Autoestima académica (IDPS)",
    "idps_convivencia"    : "Convivencia escolar (IDPS)",
    "idps_habitos"        : "Hábitos de vida (IDPS)",
    "idps_participacion"  : "Participación familiar (IDPS)",
    "C(cod_depe2)[T.2]"   : "Dependencia: Part. subvencionado",
    "C(cod_depe2)[T.3]"   : "Dependencia: Part. pagado",
    "C(cod_depe2)[T.4]"   : "Dependencia: SLEP",
    "C(cod_rural_rbd)[T.2]": "Zona: Rural",
}

tabla_mod = pd.DataFrame({
    "Variable"          : [nombres_bonitos.get(p, p) for p in modelo.params.index],
    "Coeficiente"       : modelo.params.round(3).values,
    "Error estándar"    : modelo.bse.round(3).values,
    "Valor t"           : modelo.tvalues.round(3).values,
    "Valor p"           : [f"{v:.4f}" if v >= 0.0001 else "<0.001" for v in modelo.pvalues],
    "IC 95% inf"        : modelo.conf_int()[0].round(3).values,
    "IC 95% sup"        : modelo.conf_int()[1].round(3).values,
    "Significativo"     : ["✓" if v < 0.05 else "✗" for v in modelo.pvalues],
})

display(tabla_mod)
print(f"\nR²          : {modelo.rsquared:.4f}")
print(f"R² ajustado : {modelo.rsquared_adj:.4f}")
print(f"Valor p (F) : {modelo.f_pvalue:.2e}")

## 8.3 Interpretación de coeficientes principales

**GSE (`cod_grupo`, coef. ≈ +16.1):**  
Manteniendo constantes todos los demás factores del modelo, subir un nivel en el grupo socioeconómico (por ejemplo, de Medio bajo a Medio) se asocia con un **aumento promedio de 16.1 puntos en el SIMCE**. Este coeficiente es altamente significativo (p < 0.001), lo que confirma que el GSE es el predictor más importante del rendimiento escolar.

**Convivencia escolar (`idps_convivencia`, coef. ≈ +2.0):**  
Un punto adicional en el indicador de convivencia escolar se asocia con un aumento de 2.0 puntos en el SIMCE promedio, manteniendo los demás factores constantes. Esto indica que un mejor clima escolar contribuye positivamente al rendimiento académico.

**Autoestima académica (`idps_autoestima`, coef. ≈ +0.88):**  
Un punto mayor en autoestima académica se asocia con 0.88 puntos adicionales en el promedio SIMCE. Aunque el efecto es menor al de la convivencia, sigue siendo estadísticamente significativo (p < 0.001).

**Zona rural (`C(cod_rural_rbd)[T.2]`, coef. ≈ −3.1):**  
Los establecimientos rurales obtienen, en promedio, 3.1 puntos menos en el SIMCE que los establecimientos urbanos, controlando por los demás factores. Esta diferencia es estadísticamente significativa (p ≈ 0.049).

## 8.4 Evaluación básica del modelo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Diagnóstico del Modelo de Regresión Lineal Múltiple",
             fontsize=13, fontweight="bold", y=1.02)

# Panel izquierdo: residuos vs ajustados
axes[0].scatter(modelo.fittedvalues, modelo.resid, alpha=0.3, color=COLOR_LECT, s=20, edgecolors="none")
axes[0].axhline(0, color="tomato", linewidth=1.8, linestyle="--")
axes[0].set_title("Residuos vs Valores Ajustados", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Valores ajustados (promedio SIMCE predicho)", fontsize=10)
axes[0].set_ylabel("Residuos", fontsize=10)
sns.despine(ax=axes[0])

# Panel derecho: real vs predicho
axes[1].scatter(df_modelo["promedio_simce"], modelo.fittedvalues,
                alpha=0.3, color=COLOR_MATE, s=20, edgecolors="none")
lim_min = min(df_modelo["promedio_simce"].min(), modelo.fittedvalues.min())
lim_max = max(df_modelo["promedio_simce"].max(), modelo.fittedvalues.max())
axes[1].plot([lim_min, lim_max], [lim_min, lim_max], color="tomato", linewidth=2, linestyle="--", label="Ajuste perfecto")
axes[1].set_title("Valores Reales vs Valores Predichos", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Promedio SIMCE real", fontsize=10)
axes[1].set_ylabel("Promedio SIMCE predicho", fontsize=10)
axes[1].legend(fontsize=9)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

print(f"El modelo explica el {modelo.rsquared*100:.1f}% de la varianza del promedio SIMCE (R² = {modelo.rsquared:.4f}).")
print(f"El R² ajustado es {modelo.rsquared_adj:.4f}, lo que indica un buen ajuste considerando el número de predictores.")

**Evaluación:** El modelo presenta un **R² = 0.63**, lo que significa que explica el 63% de la varianza del promedio SIMCE. Esto es un ajuste considerable para datos educativos reales, donde múltiples factores no observados (calidad docente, infraestructura, motivación individual) también influyen. Los residuos se distribuyen aproximadamente alrededor de cero, lo que sugiere que el modelo lineal es razonablemente adecuado. Como limitación, el modelo trabaja a nivel de establecimiento y no de estudiante, por lo que los coeficientes representan asociaciones a nivel agregado.

---
# 9. Discusión preliminar

Los resultados del análisis convergen en una conclusión consistente: **existe una relación positiva, fuerte y estadísticamente significativa entre el grupo socioeconómico de un establecimiento y su rendimiento en el SIMCE**.

**Hallazgos más importantes:**
1. La diferencia entre el grupo Bajo y el grupo Alto alcanza ~56 puntos en lectura y ~90 puntos en matemáticas, una brecha de más de media desviación estándar.
2. El ANOVA de Welch confirma que las diferencias entre los cinco grupos no son atribuibles al azar (F > 776, p < 0.001), y el test de Tukey demuestra que cada par de grupos se diferencia significativamente.
3. La correlación de Spearman entre GSE y promedio SIMCE es ρ = 0.72, clasificable como fuerte.
4. El modelo de regresión múltiple (R² = 0.63) muestra que el GSE tiene el mayor poder explicativo individual (~16 puntos por nivel de GSE), seguido por la convivencia escolar (~2 puntos).

**Hallazgos inesperados:** la participación familiar (IDPS) presenta un coeficiente negativo en el modelo, lo que podría indicar que las familias se involucran más en establecimientos donde los resultados son menores, o que el indicador captura dinámicas distintas a las esperadas.

**Limitaciones:**
- El análisis es a nivel de establecimiento, no de estudiante individual.
- El GSE resume condiciones socioeconómicas complejas en una sola variable ordinal.
- El modelo no incorpora factores como calidad docente, infraestructura o historial académico.

---
# 10. Próximos pasos para el avance final

| Tarea | Responsable |
|-------|-------------|
| Revisar supuestos del modelo (normalidad de residuos, multicolinealidad VIF) | Ambos |
| Explorar modelos separados para lectura y matemáticas | Luis Felipe |
| Profundizar revisión de literatura sobre brechas educativas en Chile | Nicolás |
| Incorporar análisis por región para detectar patrones territoriales | Luis Felipe |
| Mejorar visualizaciones para la presentación oral | Ambos |
| Redactar conclusión final con todos los hallazgos integrados | Ambos |

---
# 11. Referencias

- Agencia de Calidad de la Educación. (2024). *Resultados SIMCE 2° Medio 2024*. https://www.agenciaeducacion.cl
- Agencia de Calidad de la Educación. (2024). *Indicadores de Desarrollo Personal y Social 2024 (IDPS)*. https://www.agenciaeducacion.cl
- Ministerio de Educación de Chile. (2024). *Indicadores de calidad de la educación en Chile*. https://www.mineduc.cl
- Pandas Development Team. (2024). *Pandas documentation*. https://pandas.pydata.org
- Seaborn Development Team. (2024). *Seaborn documentation*. https://seaborn.pydata.org
- Statsmodels Developers. (2024). *Statsmodels documentation*. https://www.statsmodels.org
- Virtanen, P., et al. (2020). SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python. *Nature Methods*, 17, 261–272.